In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType
)
from datetime import datetime

# Audit table location
audit_path = (
    "abfss://silver@ecommercenidhi.dfs.core.windows.net/"
    "audit/pipeline_runs"
)

# Explicit schema
audit_schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("run_time", TimestampType(), True),
    StructField("rows_processed", IntegerType(), True),
    StructField("rows_rejected", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

def write_audit_log(
    pipeline_name,
    table_name,
    rows_processed,
    rows_rejected,
    status,
    error_message=None
):
    audit_data = [[
        pipeline_name,
        table_name,
        datetime.now(),
        int(rows_processed),
        int(rows_rejected),
        status,
        error_message
    ]]

    df = spark.createDataFrame(
        audit_data,
        schema=audit_schema
    )

    df.write \
        .format("delta") \
        .mode("append") \
        .save(audit_path)

    print(f"Audit logged: {status}")

In [0]:
write_audit_log(
    pipeline_name="ecommerce_pipeline",
    table_name="orders",
    rows_processed=521,
    rows_rejected=2,
    status="SUCCESS"
)

In [0]:
display(
    spark.read
    .format("delta")
    .load(audit_path)
    .orderBy("run_time", ascending=False)
)

In [0]:
from pyspark.sql.functions import broadcast

customer_orders = (
    orders
    .join(
        broadcast(customers),
        "customer_id",
        "left"
    )
)

display(customer_orders.limit(10))